# W8-T1 — English retention: what did the LoRA adaptation cost on ASVspoof 2019 LA?

This is the missing cell of W8-T1, and the first thing a reviewer will ask for.

`docs/results/gap_closure_v1.md` reports the adapter closing the code-mixed gap
(53.71% -> 1.34% EER). It does **not** report what that adaptation did to the
English performance it started from. AffectDF's Table 13 is the warning: AASIST
retrained on their data went from 0.83% to **44.52% EER** back on ASVspoof 2019 —
domain adaptation bought the new domain and destroyed the old one.

A detector that solves Hinglish by forgetting English is not a mitigation, it is a trade.

**What this notebook measures.** Three checkpoints, scored on the same 71,237-clip
ASVspoof 2019 LA eval partition, in the same session:

| Checkpoint | Code-mixed (published) | ASVspoof 2019 eval |
|---|---|---|
| `checkpoints/baseline/best.pt` (Stage-1) | 53.71% EER | 0.58% known — re-measured here as the control |
| `checkpoints/lora_codemix/best.pt` (clean adapter) | 1.34% EER | **this notebook** |
| `checkpoints/lora_codemix_channel/best.pt` (channel adapter) | 3.89% EER on the channel column | **this notebook** |

LoRA freezes the base encoder and only 1.13% of parameters moved, so there is
reason for cautious optimism that the damage is small — but that is an argument,
not a measurement, and `gap_closure_v1.md` is explicit that it must not be reported
as one.

---

## Before you run

1. **Settings -> Accelerator -> GPU T4 x2** (or P100).
2. **Settings -> Internet -> On** — needed to clone and to pull the LFS checkpoints.
3. **Add the ASVspoof 2019 dataset** (Add Input -> search "ASVspoof 2019"). The
   notebook auto-detects the `LA/` root by locating `ASVspoof2019_LA_cm_protocols`.
4. **If the repo is private:** Add-ons -> Secrets -> add a secret named `GH_PAT`
   holding a fine-grained PAT with Contents: read. The token is never printed, and
   it is stripped from the git remote as soon as the LFS pull finishes.

Roughly 45-60 min per checkpoint on a T4. Set `LIMIT` below for a smoke run first.

## 1. Configuration

In [ ]:
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH    = "main"
PAT_SECRET_NAME = "GH_PAT"      # Kaggle Secret name; ignored if the repo is public

# Smoke run first: set LIMIT = 2000, confirm the pipeline works end to end, then
# set it back to None for the full 71,237-clip eval partition.
LIMIT = None

SCORE_BASELINE = True           # re-measure Stage-1 as the control, same session
BATCH_SIZE  = 32
NUM_WORKERS = 4
MAX_SECONDS = 4.0               # must match training (configs/train_lora_codemix.yaml)

CHECKPOINTS = {
    "stage1_baseline": "checkpoints/baseline/best.pt",
    "lora_clean":      "checkpoints/lora_codemix/best.pt",
    "lora_channel":    "checkpoints/lora_codemix_channel/best.pt",
}

## 2. Clone the repo

The PAT is never echoed and never left in the remote, per the team rules doc, section 1.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"


def sh(cmd, cwd=None, check=True, quiet=False):
    """Run a command, echo it, and surface only the tail of its output."""
    if not quiet:
        print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                       text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout[-3000:])
    if r.returncode != 0:
        if r.stderr.strip():
            print(r.stderr[-3000:])
        if check:
            raise SystemExit("command failed with exit %d" % r.returncode)
    return r


token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(PAT_SECRET_NAME)
    print("using the Kaggle secret %r for git auth" % PAT_SECRET_NAME)
except Exception:
    print("no Kaggle secret found - assuming the repository is public")


def scrub(text):
    """Never let the PAT reach the notebook output."""
    return text.replace(token, "***") if token else text


if REPO.exists():
    shutil.rmtree(REPO)

url = "https://%s@%s" % (token, REPO_HOST) if token else "https://%s" % REPO_HOST
# This one command is deliberately NOT echoed - the URL carries the token.
r = subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, str(REPO)],
                   text=True, capture_output=True)
if r.returncode != 0:
    print(scrub(r.stderr)[-3000:])
    raise SystemExit("clone failed - check the PAT secret and that Internet is On")
print("cloned %s ->" % BRANCH, REPO)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
sh("git log --oneline -3")

## 3. Pull the checkpoints (Git LFS)

The three `best.pt` files are 134-byte LFS pointers in the tree. Without this step
`torch.load` fails on a pointer file rather than on a checkpoint, which is a
confusing error to debug 40 minutes into a run.

In [ ]:
sh("apt-get -qq update && apt-get -qq install -y git-lfs", check=False)
sh("git lfs install --local")
sh("git lfs pull --include 'checkpoints/**'", check=False)

# Strip the token now that LFS is done: a PAT is injected inline, never stored.
sh("git remote set-url origin https://%s" % REPO_HOST, quiet=True)
print("remote scrubbed of credentials")

ok = True
for name, rel in CHECKPOINTS.items():
    p = REPO / rel
    size = p.stat().st_size if p.exists() else 0
    if size > 10_000_000:
        status = "OK"
    elif size == 0:
        status = "MISSING"
    else:
        status = "LFS POINTER - not fetched"
    print("%-18s %8.1f MB  %s" % (name, size / 1e6, status))
    if status != "OK" and (name != "stage1_baseline" or SCORE_BASELINE):
        ok = False
if not ok:
    raise SystemExit("a checkpoint did not come down - re-run this cell")

## 4. Dependencies

Kaggle's preinstalled torch is left alone on purpose. The college-PC run used
torch 2.8.0+cu128 and the Kaggle channel repro used 2.10.0+cu128; both worked, and
`channel_matched_v1.md` already attributes a 3.5 pp clean-column difference to that
environment gap. Forcing a downgrade here would be a bigger change than the one
being measured.

In [ ]:
sh("%s -m pip install -q 'transformers>=4.57,<5' soundfile==0.12.1 librosa==0.11.0"
   % sys.executable, check=False)

import torch

print("torch     ", torch.__version__)
print("cuda avail", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("no GPU - set Accelerator to GPU T4 x2 and restart the session")
print("device    ", torch.cuda.get_device_name(0),
      "(%.1f GB)" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 5. Locate ASVspoof 2019 LA and build the eval manifest

`data/manifests/asvspoof_*.csv` is gitignored — the paths are absolute and
machine-specific — so the manifest is rebuilt here from the attached dataset.

In [ ]:
protocols = glob.glob("/kaggle/input/**/ASVspoof2019_LA_cm_protocols", recursive=True)
if not protocols:
    print("contents of /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print("  ", p)
    raise SystemExit("ASVspoof2019_LA_cm_protocols not found - attach the ASVspoof 2019 dataset")

LA_ROOT = Path(protocols[0]).parent
print("LA root:", LA_ROOT)

flac = LA_ROOT / "ASVspoof2019_LA_eval" / "flac"
if not flac.is_dir():
    raise SystemExit("eval flac directory missing: %s" % flac)
print("eval flac dir:", flac)

sh("%s -m src.data.asvspoof --la-root '%s' --splits eval --out-dir data/manifests"
   % (sys.executable, LA_ROOT))

In [ ]:
import pandas as pd

MANIFEST = REPO / "data/manifests/asvspoof_eval.csv"
man = pd.read_csv(MANIFEST)
print("rows            {:,}".format(len(man)))
print("bonafide/spoof  {:,} / {:,}".format((man.label == "bonafide").sum(),
                                           (man.label == "spoof").sum()))
print("speakers        %d" % man.speaker.nunique())
print("attacks         %s" % sorted(a for a in man.tool.unique() if a != "none"))

# The published protocol figures, re-checked rather than trusted. If these do not
# match, the manifest is wrong and every number below is meaningless.
assert len(man) == 71237, "expected 71,237 eval clips, got {:,}".format(len(man))
assert man.speaker.nunique() == 67, "expected 67 eval speakers, got %d" % man.speaker.nunique()
print("\nmatches the published eval protocol: 71,237 clips / 67 speakers")

## 6. Anti-leakage checklist

The team rules doc, section 6: `tests/test_splits.py` runs before every training launch. Running
it here too — a failure means the golden dataset rule has been broken somewhere
upstream, and no number from this notebook should be trusted.

In [ ]:
r = sh("%s -m pytest tests/test_splits.py tests/test_lora.py -q" % sys.executable, check=False)
if r.returncode != 0:
    print("\n!! the anti-leakage or LoRA tests did not pass - read the output above "
          "before trusting anything below")

## 7. Score each checkpoint on ASVspoof 2019 LA eval

`--partial` writes an incremental score cache, so an interrupted session resumes
instead of restarting the 71,237 clips.

`evaluate.py` rebuilds the LoRA wrapping from the checkpoint's own key names (the
rank comes from `lora_A`'s shape), so the adapted checkpoints load with no extra
flags and a Stage-1 checkpoint is left untouched.

In [ ]:
OUT = REPO / "experiments"
OUT.mkdir(exist_ok=True)

targets = list(CHECKPOINTS.items())
if not SCORE_BASELINE:
    targets = [(n, p) for n, p in targets if n != "stage1_baseline"]

results = {}
for name, ckpt in targets:
    print("\n" + "=" * 72)
    print("scoring %s  (%s)" % (name, ckpt))
    print("=" * 72)
    out_json = OUT / ("asvspoof_retention_%s.json" % name)
    cmd = (
        "%s -m src.training.evaluate "
        "--checkpoint '%s' --manifest '%s' "
        "--device cuda --batch-size %d --num-workers %d --max-seconds %s "
        "--partial '%s/_partial_%s.csv' "
        "--scores-out '%s/asvspoof_retention_%s_scores.csv' "
        "--out '%s'"
        % (sys.executable, ckpt, MANIFEST, BATCH_SIZE, NUM_WORKERS, MAX_SECONDS,
           OUT, name, OUT, name, out_json)
    )
    if LIMIT:
        cmd += " --limit %d" % LIMIT
    t0 = time.time()
    sh(cmd)
    print("[%s] took %.1f min" % (name, (time.time() - t0) / 60))
    results[name] = json.loads(out_json.read_text())

## 8. The retention table

In [ ]:
# Code-mixed numbers as published, for the other half of each row.
CODEMIX = {
    "stage1_baseline": ("53.71%", "0.459"),
    "lora_clean":      ("1.34%", "0.9997"),
    "lora_channel":    ("13.92% clean / 3.89% channel", "0.924 / 0.993"),
}


def pooled(d):
    p = d.get("pooled", d)
    return p.get("eer"), p.get("auc")


rows = []
for name, _ in targets:
    eer, auc = pooled(results[name])
    rows.append({
        "checkpoint": name,
        "ASVspoof EER": "{:.2f}%".format(eer * 100) if isinstance(eer, float) else eer,
        "ASVspoof AUC": "{:.4f}".format(auc) if isinstance(auc, float) else auc,
        "code-mixed EER (published)": CODEMIX.get(name, ("-", "-"))[0],
    })

print(pd.DataFrame(rows).to_string(index=False))

if SCORE_BASELINE and "stage1_baseline" in results and "lora_clean" in results:
    b, _ = pooled(results["stage1_baseline"])
    a, _ = pooled(results["lora_clean"])
    delta = (a - b) * 100
    print("\nEnglish retention cost (clean adapter): "
          "{:.2f}% -> {:.2f}% EER   ({:+.2f} pp)".format(b * 100, a * 100, delta))
    print("\nFor scale: AffectDF Table 13 reports AASIST going 0.83% -> 44.52% EER "
          "(+43.69 pp) after domain adaptation.")
    if delta < 2:
        print("VERDICT: retention holds. LoRA bought the new domain without destroying "
              "the old one - the freeze-the-encoder argument is now a measurement.")
    elif delta < 10:
        print("VERDICT: measurable but modest degradation. Report it as a trade, with "
              "the size stated.")
    else:
        print("VERDICT: substantial forgetting. This is the AffectDF failure mode and it "
              "belongs in the paper as a limitation, not a footnote.")

## 9. Per-attack breakdown

A pooled EER hides what ASVspoof was built to expose: the eval partition holds 13
attacks (A07-A19) that never appear in train or dev. A detector can look fine pooled
and fail one unfamiliar family completely.

In [ ]:
for name, _ in targets:
    attacks = results[name].get("per_attack")
    if not attacks:
        continue
    df = pd.DataFrame(attacks)
    print("\n=== %s - worst attacks first ===" % name)
    print(df.head(8).to_string(index=False))

## 10. Collect the artefacts

Download these from the notebook's **Output** tab, then commit them from a machine
that has the git identities (the team rules doc, section 1 — W8-T1 is M's task, so it commits
under M). Check `.gitignore` before committing the score CSVs.

In [ ]:
for p in sorted(OUT.glob("asvspoof_retention_*")):
    dest = WORK / p.name
    if p.resolve() != dest.resolve():
        shutil.copy(p, dest)
    print("%s  (%.2f MB)" % (dest, dest.stat().st_size / 1e6))

summary = {
    "task": "W8-T1 english retention",
    "manifest": "asvspoof_eval.csv (71,237 clips, 67 speakers)",
    "limit": LIMIT,
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "results": {k: {"eer": pooled(v)[0], "auc": pooled(v)[1]} for k, v in results.items()},
    "codemix_reference": CODEMIX,
}
(WORK / "asvspoof_retention_summary.json").write_text(json.dumps(summary, indent=2))
print("\n" + json.dumps(summary, indent=2))